# 1. Import library

In [ ]:
import os
import torch
import random
import shutil
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# 2. Setup

### 2.1. PATH

Before running the training code, place the dataset that you want to use for training the SRGAN model inside the **dataset** folder.

In [ ]:
DATASET_PATH = f"dataset"

SRGAN_PATH = f"srgan"
TRAIN_PATH = f"{SRGAN_PATH}/train"
VAL_PATH = f"{SRGAN_PATH}/val"

def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"Created folder: {path}")
    else:
        print(f"Folder exists: {path}")

ensure_dir(TRAIN_PATH)
ensure_dir(VAL_PATH)

print("\nFinal Structure:")
print("Dataset:", DATASET_PATH)
print(" ├── train:", TRAIN_PATH)
print(" └── val:", VAL_PATH)

### 2.2. Config

Adjust the **high_res_size** variable to match the resolution of the input images in the **dataset** folder.

In [ ]:
high_res_size = 224
scale = 4
low_res_size = high_res_size // scale

print("high resolution :", high_res_size)
print("low resolution :", low_res_size)

# 3. Split Train-Valid for Training SRGAN

In [ ]:
TRAIN_HR = os.path.join(TRAIN_PATH, "high_res")
TRAIN_LR = os.path.join(TRAIN_PATH, "low_res")
VAL_HR   = os.path.join(VAL_PATH, "high_res")
VAL_LR   = os.path.join(VAL_PATH, "low_res")

for path in [TRAIN_HR, TRAIN_LR, VAL_HR, VAL_LR]:
    os.makedirs(path, exist_ok=True)

image_files = [f for f in os.listdir(DATASET_PATH)
               if f.lower().endswith((".png", ".jpg", ".jpeg"))]

print("Total images:", len(image_files))

In [ ]:
train_files, val_files = train_test_split(
    image_files,
    test_size=0.1  # @param{type:"number"}
    ,random_state=42,
    shuffle=True
)

def copy_files(file_list, target_dir, mode):
    for f in tqdm(file_list, desc=f"{mode.upper()} HR"):
        shutil.copy(
            os.path.join(DATASET_PATH, f),
            os.path.join(target_dir, f)
        )

copy_files(train_files, TRAIN_HR, mode="train")
copy_files(val_files, VAL_HR, mode="val")

In [ ]:
def generate_lr(hr_dir, lr_dir, mode):
    files = os.listdir(hr_dir)

    for f in tqdm(files, desc=f"{mode.upper()} LR"):
        hr_path = os.path.join(hr_dir, f)
        lr_path = os.path.join(lr_dir, f)

        img = Image.open(hr_path).convert("RGB")
        w, h = img.size

        w = (w // scale) * scale
        h = (h // scale) * scale
        img = img.crop((0, 0, w, h))

        lr_img = img.resize(
            (w // scale, h // scale),
            Image.BICUBIC
        )
        lr_img.save(lr_path)

generate_lr(TRAIN_HR, TRAIN_LR, mode="train")
generate_lr(VAL_HR, VAL_LR, mode="val")

In [ ]:
sample_file = random.choice(os.listdir(TRAIN_LR))

lr_path = os.path.join(TRAIN_LR, sample_file)
hr_path = os.path.join(TRAIN_HR, sample_file)

lr_img = Image.open(lr_path)
hr_img = Image.open(hr_path)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(lr_img)
plt.title("LR Image")

plt.subplot(1,2,2)
plt.imshow(hr_img)
plt.title("HR Image")

plt.show()

print(f"\nSample file: {sample_file}")
print("LR size:", lr_img.size)
print("HR size:", hr_img.size)